Section 7.3: LIME: Local Interpretable Model-agnostic Explanations. Learn how to use LIME to explain individual predictions made by a text classification model. This project builds a sentiment classifier with TF-IDF and Logistic Regression, then uses LIME to identify the words and phrases that most influenced each prediction, providing local, human-interpretable explanations for black-box machine learning models.


In [2]:
# -------------------------------------------------------
# If needed, install the required libraries:
#
!pip install lime
!pip install scikit-learn
!pip install numpy
# -------------------------------------------------------

# Import the required libraries.
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from lime.lime_text import LimeTextExplainer

# -------------------------------------------------------
# Step 1: Create a small training dataset.
#
# 1 = Positive review
# 0 = Negative review
#
# In a real project, this dataset would typically contain
# thousands of reviews instead of just a few examples.
# -------------------------------------------------------

train_texts = [
    "amazing clever thrilling wonderful masterpiece",
    "excellent captivating brilliant enjoyable inspiring",
    "well developed characters clever twists",
    "beautiful storytelling highly recommend",
    "boring dull predictable waste disappointing",
    "terrible slow weak confusing clumsy",
    "bad ending no sense forgettable",
    "uninspired repetitive flat awful"
]

train_labels = [1, 1, 1, 1, 0, 0, 0, 0]

# -------------------------------------------------------
# Step 2: Build the machine-learning pipeline.
#
# A pipeline automatically performs multiple steps in order.
#
# First:
#     Convert text into numerical TF-IDF features.
#
# Then:
#     Train a Logistic Regression classifier.
# -------------------------------------------------------

text_classifier = make_pipeline(

    # Convert words into numerical features.
    #
    # lowercase=True
    #     Converts all text to lowercase so that
    #     "Amazing" and "amazing" are treated as
    #     the same word.
    #
    # stop_words="english"
    #     Removes very common words such as:
    #     "the", "and", "is", "was", etc.
    #
    #     These words usually carry very little
    #     meaning and often dominate explanations.
    #     Removing them allows LIME to focus on
    #     informative words like "amazing",
    #     "boring", or "clever".
    #
    # ngram_range=(1,2)
    #     Use both:
    #       • Single words (unigrams)
    #       • Two-word phrases (bigrams)
    #
    #     For example:
    #       "great"
    #       "great story"
    #
    TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2)
    ),

    # Logistic Regression is our classifier.
    #
    # random_state=42
    #     Makes the results reproducible.
    #
    # class_weight="balanced"
    #     Automatically adjusts for any imbalance
    #     between positive and negative examples.
    #
    LogisticRegression(
        random_state=42,
        class_weight="balanced"
    )
)

# -------------------------------------------------------
# Step 3: Train the classifier.
#
# The model learns which words are associated with
# positive and negative reviews.
# -------------------------------------------------------

text_classifier.fit(train_texts, train_labels)

# -------------------------------------------------------
# Step 4: A new review we want to explain.
# -------------------------------------------------------

new_review = (
    "Amazing story with clever twists "
    "and well developed characters."
)

# Predict whether the review is positive or negative.
prediction = text_classifier.predict([new_review])[0]

# -------------------------------------------------------
# Step 5: Create the LIME explainer.
# -------------------------------------------------------

explainer = LimeTextExplainer(

    # Human-readable names for the classes.
    class_names=["Negative", "Positive"],

    # Makes LIME produce reproducible results.
    random_state=42
)

# -------------------------------------------------------
# Step 6: Explain the prediction.
# -------------------------------------------------------

explanation = explainer.explain_instance(

    # The text we want explained.
    new_review,

    # Function used by LIME to obtain prediction
    # probabilities from the trained model.
    #
    # LIME repeatedly creates modified versions
    # of the review and calls this function to
    # observe how the probabilities change.
    text_classifier.predict_proba,

    # Show the six most influential words
    # or phrases.
    num_features=6,

    # Explain only the predicted class.
    #
    # Without this parameter, LIME may generate
    # explanations for multiple classes.
    labels=[prediction]
)

# -------------------------------------------------------
# Step 7: Display the prediction.
# -------------------------------------------------------

print(
    f"Model prediction: "
    f"{'Positive' if prediction == 1 else 'Negative'}"
)

print("-" * 50)
print("Most influential words:")
print("-" * 50)

# -------------------------------------------------------
# explanation.as_list()
#
# Returns a list of:
#
# (feature, weight)
#
# A positive weight pushes the prediction toward
# the predicted class.
#
# A negative weight pushes the prediction away
# from the predicted class.
# -------------------------------------------------------

for word, weight in explanation.as_list(label=prediction):

    print(f"{word:20s} {weight:+.4f}")

# -------------------------------------------------------
# Optional (Jupyter Notebook / Google Colab):
#
# Displays an interactive visualization showing
# how each word influenced the prediction.
#
# explanation.show_in_notebook()
# -------------------------------------------------------


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=e3153272c811d38981a70e89a33bad367fc3140a452b0401e78146688fd0c153
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime
Model prediction: Positive
--------------------------------------------------
Most influential words:
--------------------------------------------------
clever               +0.0274
characters           +0.0156
developed            +0.0156
twists               +0.0154
Amazing              +0.0085
story                -0.0023
